In [1]:
from src.rqvae import RQVAE, RQVAELoss
from pathlib import Path
import torch
import torch.nn as nn
import torch.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

In [2]:
torch.manual_seed(42)
np.random.seed(42)

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


### Downloading data

In [4]:
path = Path('../data/processed/item_features')

title_path = path / 'title_svd128.npy'
year_path = path / 'year_minmax.npy'
genres_path = path / 'genres.npy'

title = np.load(title_path)
year = np.load(year_path)
genres = np.load(genres_path)

X = np.concatenate([title, year, genres], axis=-1)
X = torch.tensor(X).to(device)

### Downloading model parameters

In [5]:
path = Path('../runs')

In [6]:
RQVAE_baseline_path = path / 'baseline_rqvae'

baseline_config_path = RQVAE_baseline_path/ 'config.json'
baseline_last_checkpoint_path = RQVAE_baseline_path / 'checkpoint_last.pt'

baseline_config = json.load(open(baseline_config_path, 'r'))
baseline_last_checkpoint = torch.load(baseline_last_checkpoint_path)

In [7]:
RQVAE_advanced_path = path / 'RQVAE_ProgressiveMasking_ContrastiveRegularization'

advanced_config_path = RQVAE_advanced_path / 'config.json'
advanced_last_checkpoint_path = RQVAE_advanced_path / 'checkpoint_last.pt'

advanced_config = json.load(open(advanced_config_path, 'r'))
advanced_last_checkpoint = torch.load(advanced_last_checkpoint_path)

### Overwiev of the parameters of both models

In [8]:
df_baseline = pd.DataFrame([baseline_config]).drop('variant', axis=1).T
df_advanced = pd.DataFrame([advanced_config]).drop('variant', axis=1).T
df = pd.concat([df_baseline, df_advanced], axis=1)
df.columns = ['RQVAE', 'RQVAE + Progressive Masking + Contrastive Regularization']
df

,RQVAE,RQVAE + Progressive Masking + Contrastive Regularization
in_d,147,147
h_d,64,64
n_levels,3,3
codebook_sizes,"[2048, 1024, 512]","[2048, 1024, 512]"
lr,0.005,0.005
beta,0.001,0.001
epochs,400,400
batch_size,512,512
device,cuda,cuda


### Creating models

In [9]:
baseline_model = RQVAE(
    in_d=baseline_config['in_d'],
    h_d=baseline_config['h_d'],
    n_levels=baseline_config['n_levels']
).to(device)

baseline_model.load_state_dict(baseline_last_checkpoint['model'])
baseline_model.eval()
print("Loaded. device:", device)

Loaded. device: cuda


In [10]:
advanced_model = RQVAE(
    in_d=advanced_config['in_d'],
    h_d=advanced_config['h_d'],
    n_levels=advanced_config['n_levels']
).to(device)

advanced_model.load_state_dict(advanced_last_checkpoint['model'])
advanced_model.eval()
print("Loaded. device:", device)

Loaded. device: cuda


### Evaluation

In [11]:
with torch.no_grad():
    baseline_x_hat, baseline_h, baseline_z_q, baseline_SIDs, baseline_r_list, baseline_q_list = baseline_model(X)
    baseline_SIDs  = baseline_SIDs.to('cpu').numpy()
    
    advanced_x_hat, advanced_h, advanced_z_q, advanced_SIDs, advanced_r_list, advanced_q_list = advanced_model(X)
    advanced_SIDs  = advanced_SIDs.to('cpu').numpy()

    loss_func = RQVAELoss()
    total_b, recon_b, cb_b, com_b, con_b = loss_func(
        X, baseline_x_hat, baseline_r_list, baseline_q_list
    )

    total_a, recon_a, cb_a, com_a, con_a = loss_func(
        X, advanced_x_hat, advanced_r_list, advanced_q_list
    )

### Metrics

In [16]:
unique_sids_baseline = len(np.unique(baseline_SIDs, axis=0))
unique_sids_advanced = len(np.unique(advanced_SIDs, axis=0))
n = X.shape[0]
 
data = [
    [
        'baseline',
        f'{round(unique_sids_baseline / n * 100, 2)} %',
        round(total_b.item(), 4),
        round(recon_b.item(), 4),
        round(cb_b.item(), 4),
        round(com_b.item(), 4),
    ],
    [
        'advanced',
        f'{round(unique_sids_advanced / n * 100, 2)} %',
        round(total_a.item(), 4),
        round(recon_a.item(), 4),
        round(cb_a.item(), 4),
        round(com_a.item(), 4),
    ]
]


df_m = pd.DataFrame(
    columns=['SID Model', 'SID Uniqueness', 'Total Loss', 'Recon Loss', 'Codebook Loss', 'Commit Loss'],
    data=data
).set_index('SID Model')

df_m

,SID Uniqueness,Total Loss,Recon Loss,Codebook Loss,Commit Loss
SID Model,,,,,
baseline,16.81 %,0.9599,0.0015,0.6390,0.3195
advanced,39.69 %,2.2364,0.0021,1.4895,0.7448
